# Sarvam-1 SFT on Odia GSM8K

Supervised fine-tuning of `sarvamai/sarvam-1` on the `train` split of `tripathysagar/odia-gsm8k`.

- TRL `SFTTrainer` with **QLoRA** (4-bit) for single-GPU training on RunPod.
- Training metrics → **Comet ML** (set `COMET_API_KEY` to enable).
- Sanity-check rollouts traced to **Opik** (optional, set `OPIK_API_KEY`).
- Final adapter merged into the base model and pushed to the HF Hub.

## 1. Setup

In [ ]:
# %pip install -q --upgrade transformers trl peft bitsandbytes accelerate datasets python-dotenv comet_ml opik huggingface_hub

In [ ]:
import gc
import os
import time
from pathlib import Path

# Put HF cache on the RunPod persistent volume so model weights survive pod restart
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")

# Import comet_ml *before* torch so Comet can auto-instrument the framework.
# (Comet warns and disables some autologging if torch is imported first.)
import comet_ml

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from huggingface_hub import login as hf_login, create_repo

print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Configuration

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable so `config.py` resolves whether this notebook's
# working dir is the repo root or the notebooks/ subfolder.
_root = Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import *   # all non-secret params (config.py also loads .env secrets)

# Local aliases — keep the names the cells below already use
BASE_MODEL_ID    = MODEL_ID
COMET_PROJECT    = COMET_PROJECT_NAME
COMET_TAGS_EXTRA = COMET_TAGS

# Fail fast on missing push target — don't waste hours of training
if PUSH_TO_HUB:
    assert SFT_HUB_MODEL_ID, (
        "PUSH_TO_HUB=True but neither SFT_HUB_MODEL_ID nor HF_USERNAME is set. "
        "Set one in config.py before training, or set PUSH_TO_HUB=False."
    )

SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base model : {BASE_MODEL_ID}")
print(f"Dataset    : {DATASET_ID} ({TRAIN_SPLIT})")
print(f"Output dir : {SFT_OUTPUT_DIR}")
print(f"Hub repo   : {SFT_HUB_MODEL_ID or '(unset)'}  push={PUSH_TO_HUB}")
print(f"QLoRA      : {USE_QLORA}  GPU: {GPU_TYPE or '(not set)'}")
print(f"Epochs={NUM_EPOCHS}  lr={LEARNING_RATE}  bs={BATCH_SIZE}x{GRAD_ACCUM}  max_seq={MAX_SEQ_LEN}")

## 3. Authenticate Hub + Tracking

In [ ]:
if HF_TOKEN:
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF Hub : logged in")
else:
    print("HF Hub : HF_TOKEN not set — push will fail")

if COMET_API_KEY:
    tags = ["sft", "qlora" if USE_QLORA else "bf16"]
    if GPU_TYPE:
        tags.append(GPU_TYPE)
    if COMET_TAGS_EXTRA:
        tags.extend(t.strip() for t in COMET_TAGS_EXTRA.split(",") if t.strip())
    os.environ["COMET_MODE"]         = "ONLINE"
    os.environ["COMET_PROJECT_NAME"] = COMET_PROJECT
    os.environ["COMET_TAGS"]         = ",".join(tags)
    if COMET_WORKSPACE:
        os.environ["COMET_WORKSPACE"] = COMET_WORKSPACE
    comet_ml.login(api_key=COMET_API_KEY)   # auth only — project/workspace/tags set via env vars above
    REPORT_TO = "comet_ml"
    print(f"Comet  : configured  project={COMET_PROJECT}  tags={tags}")
else:
    REPORT_TO = "none"
    print("Comet  : not configured (set COMET_API_KEY to enable)")

# Opik for LLM-trace logging (used in sanity-check rollouts below)
OPIK_ENABLED = bool(OPIK_API_KEY)
if OPIK_ENABLED:
    import opik
    os.environ["OPIK_PROJECT_NAME"] = OPIK_PROJECT_NAME   # Opik reads this automatically
    opik.configure(api_key=OPIK_API_KEY, workspace=OPIK_WORKSPACE)
    print(f"Opik   : configured  project={OPIK_PROJECT_NAME}")
else:
    print("Opik   : not configured (optional)")

## 4. Load & Format Dataset

In [ ]:
ds = load_dataset(DATASET_ID, split=TRAIN_SPLIT, token=HF_TOKEN)
print(f"Loaded {len(ds)} training examples")
print("Columns:", ds.column_names)
print("\nSample:")
print(ds[0])

In [ ]:
SYSTEM_PROMPT = (
    "ଆପଣ ଜଣେ ସହାୟକ ଗଣିତ ସହକାରୀ ଅଟନ୍ତି। "
    "ତଳେ ଦିଆଯାଇଥିବା ସମସ୍ୟାକୁ ପର୍ଯ୍ୟାୟକ୍ରମେ ସମାଧାନ କରନ୍ତୁ। "
    "ଶେଷରେ, ଆପଣଙ୍କର ଚୂଡ଼ାନ୍ତ ସାଂଖ୍ୟିକ ଉତ୍ତରକୁ ଏକ ନୂଆ ଧାଡ଼ିରେ '####' ସହିତ ଆରମ୍ଭ କରି ଲେଖନ୍ତୁ।"
)
print("SYSTEM_PROMPT defined. Dataset will be formatted after tokenizer load.")

## 5. Load Model + Tokenizer

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model_kwargs = dict(
    token=HF_TOKEN,
    trust_remote_code=True,
    device_map="auto",
)

if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = bnb_config
else:
    model_kwargs["torch_dtype"] = torch.bfloat16

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)
model.config.use_cache = False  # incompatible with gradient checkpointing

if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
print("Model ready.")

In [ ]:
EOS = tokenizer.eos_token

def format_example(example):
    q = example[QUESTION_COL]
    a = example[ANSWER_COL]
    return {"text": f"{SYSTEM_PROMPT}\n\nପ୍ରଶ୍ନ: {q}\nଉତ୍ତର: {a}{EOS}"}

train_ds = ds.map(format_example, remove_columns=ds.column_names)
print(f"Formatted {len(train_ds)} examples.")
print("\nFirst formatted sample (truncated):")
print(train_ds[0]["text"][:600], "...")

## 6. Configure Trainer

In [ ]:
run_name = "-".join(filter(None, ["sarvam1-sft", GPU_TYPE, str(int(time.time()))]))

sft_config = SFTConfig(
    output_dir=str(SFT_OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field="text",
    report_to=REPORT_TO,
    run_name=run_name,
    push_to_hub=False,    # we merge + push manually below
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainable, total = trainer.model.get_nb_trainable_parameters()
print(f"Train examples   : {len(train_ds)}")
print(f"Trainable params : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"Run name         : {run_name}")

## 7. Train

In [ ]:
trainer.train()
print("Training complete.")

adapter_dir = SFT_OUTPUT_DIR / "final-adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter saved to {adapter_dir}")

## 8. Merge LoRA → Push to HF Hub

In [ ]:
merged_model = None

if PUSH_TO_HUB:
    # Free the quantized training model before reloading in bf16 — gc.collect() is required
    # because del alone doesn't release LoRA / accelerator wrapper references quickly enough
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    print(f"Reloading base model in bf16 to merge adapter...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    merged_model = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()

    print(f"Pushing merged model → {SFT_HUB_MODEL_ID}")
    create_repo(SFT_HUB_MODEL_ID, token=HF_TOKEN, private=PRIVATE_REPO, exist_ok=True)
    merged_model.push_to_hub(SFT_HUB_MODEL_ID, token=HF_TOKEN, private=PRIVATE_REPO)
    tokenizer.push_to_hub(SFT_HUB_MODEL_ID, token=HF_TOKEN, private=PRIVATE_REPO)
    print(f"Pushed: https://huggingface.co/{SFT_HUB_MODEL_ID}")
else:
    print("PUSH_TO_HUB=false — adapter saved locally only.")

## 9. Sanity-Check Inference

In [ ]:
test_ds = load_dataset(DATASET_ID, split="test", token=HF_TOKEN).select(range(3))

eval_model = merged_model if merged_model is not None else trainer.model
eval_model.eval()

_track = opik.track(name="sft_sanity_check") if OPIK_ENABLED else (lambda f: f)

@_track
def run_sample(question: str) -> str:
    prompt = f"{SYSTEM_PROMPT}\n\nପ୍ରଶ୍ନ: {question}\nଉତ୍ତର:"
    inputs = tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

for ex in test_ds:
    pred = run_sample(ex[QUESTION_COL])
    print("Q   :", ex[QUESTION_COL][:120], "...")
    print("Gold:", str(ex[ANSWER_COL])[:200], "...")
    print("Pred:", pred[:300])
    print("-" * 60)

## 10. Finalize Tracking

In a Jupyter environment Comet does not auto-close the run, so we end it
explicitly to flush all metrics, parameters and code before exiting.

In [ ]:
# Close the Comet experiment so all data is logged before the kernel exits.
if REPORT_TO == "comet_ml":
    comet_ml.end()
    print("Comet  : experiment ended — all metrics & code flushed")